<a href="https://colab.research.google.com/github/shreyaghora/ML/blob/main/ML_Result01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Import  all the necessary libraries,functions, models


In [ ]:
!pip install xgboost

In [ ]:
!pip install imblearn

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier


from imblearn.over_sampling import SMOTE, ADASYN

Loading the dataset from .csv to pandas's dataframe

In [ ]:
df = pd.read_csv('/content/bank-full.csv', sep=';')

Preprocessing, Scaling, Sampling

In [ ]:
# Droping duration column to prevent leakage
df.drop('duration', axis=1, inplace=True) #drop method used for removing specified rows and column from a dataset


# For removing unknown values like 0, null, ?
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


#Encoding in binary[0 & 1] values i.e. Final value or, Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})


# One-Hot Encoding: Convert ordinary data into numeric form [Only 1 column is (hot)1 at a time in each row for N categories]
df = pd.get_dummies(df, drop_first=True)


# Split Features(X:Input variables used for train the model and make prediction) & Target(y:Variable the model is intend to predict)

X = df.drop('y', axis=1)
y = df['y']


#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


#Scaling: Step of preprocessing
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


#Sampling Techniques
smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train, y_train)


#Models
models = {

    "Decision Tree": DecisionTreeClassifier
     (
            max_depth=8,
            min_samples_split=10,
            min_samples_leaf=4,
            random_state=42

    ),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=300),

   "MLP": MLPClassifier(
       max_iter=500, random_state=42, early_stopping=True,hidden_layer_sizes=(128,64)
                  ),
    "XGBoost": XGBClassifier( eval_metric='logloss', random_state=42),


}

# Evaluation Metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


#Cross Validation Setup
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
#For storing
results= []
#Function to Evaluate Models
def evaluate_models(X, y, label):
    print(f"\n===== {label} =====")
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        metrics={
            "Label":f"{label}",
            "Model": name,
            "Accuracy":  f"{np.mean(scores['test_accuracy']):.4f}",
            "Precision": f"{np.mean(scores['test_precision']):.4f}",
            "Recall":    f"{np.mean(scores['test_recall']):.4f}",
            "F1 Score":  f"{np.mean(scores['test_f1']):.4f}",
            "ROC-AUC":   f"{np.mean(scores['test_roc_auc']):.4f}"
        }
        results.append(metrics)

        print(f"\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")

#Run Experiments

# Baseline(Without sampling)
evaluate_models(X_train, y_train, "Baseline (No Sampling)")


#(With Sampling)

# Result of using SMOTE algo.
evaluate_models(X_train_sm, y_train_sm, "SMOTE")

# Result of using ADASYN algo.
evaluate_models(X_train_ad, y_train_ad, "ADASYN")
#Converting into dataframe
results_df = pd.DataFrame(results)
from google.colab import drive
# Save directly to Drive
results_df.to_excel('/content/drive/MyDrive/results.xlsx', index=False)


===== Baseline (No Sampling) =====

Model: Decision Tree
Accuracy:  0.8921
Precision: 0.6226
Recall:    0.1988
F1 Score:  0.3009
ROC-AUC:   0.7034

Model: Random Forest
Accuracy:  0.8937
Precision: 0.6339
Recall:    0.2156
F1 Score:  0.3213
ROC-AUC:   0.7671

Model: MLP
Accuracy:  0.8932
Precision: 0.6263
Recall:    0.2198
F1 Score:  0.3234
ROC-AUC:   0.7609

Model: XGBoost
Accuracy:  0.8904
Precision: 0.5727
Recall:    0.2486
F1 Score:  0.3462
ROC-AUC:   0.7703

===== SMOTE =====

Model: Decision Tree
Accuracy:  0.8041
Precision: 0.8585
Recall:    0.7292
F1 Score:  0.7882
ROC-AUC:   0.8703

Model: Random Forest
Accuracy:  0.9368
Precision: 0.9534
Recall:    0.9186
F1 Score:  0.9357
ROC-AUC:   0.9815

Model: MLP
Accuracy:  0.8728
Precision: 0.8606
Recall:    0.8899
F1 Score:  0.8749
ROC-AUC:   0.9404

Model: XGBoost
Accuracy:  0.9320
Precision: 0.9694
Recall:    0.8922
F1 Score:  0.9292
ROC-AUC:   0.9707

===== ADASYN =====

Model: Decision Tree
Accuracy:  0.8224
Precision: 0.8822
Rec